In [32]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sktime.split import ExpandingWindowSplitter
from lightgbm import LGBMRegressor
import joblib
from catboost import CatBoostRegressor
import json
import optuna
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from features.FeatureExtractor import FeatureExtractor
from features.TargetEncoder import TargetEncoder

In [33]:
cleaned_dataset_address = "dataset/interim/past_dataset.csv"

In [34]:
SEED = 87
TARGET = "general_dam_occupancy_rate"

In [35]:
try:
    with open("models/best_features_by_lightgbm_with_te.json", errors="FileNotFoundError") as f:
        best_features = json.load(f)
        
except FileNotFoundError:
    best_features = []

In [36]:
past_knowledge = (
    pd.read_csv(
        cleaned_dataset_address,
        parse_dates=["datetime"],
        converters={"weather_code": str},
    )
    .set_index("datetime")
    .sort_index()
)


In [37]:
parameters = {
    "past_knowledge": past_knowledge,
    "cyclical_feature_names": {
        "month": 12,
        "day": 31,
        "day_of_year": 365,
        "week_of_year": 52,
        "quarter": 4,
        # "season": 4,
        "is_weekend": 2,
        "precipitation_hours": 24,
    },
    "lag_size": 30,
    "window_size": 30,
}

In [38]:
feature_extractor = FeatureExtractor(**parameters)

In [39]:
known_dates = past_knowledge.index

In [40]:
y_values = past_knowledge.loc[:, [TARGET]]

In [41]:
if best_features:
    X_values = feature_extractor.transform(known_dates).filter(items=[*best_features])
else:
    X_values = feature_extractor.transform(known_dates)

In [42]:
train_size = int(len(X_values) * 0.8)

In [43]:
train_df = X_values.iloc[:train_size].merge(y_values, on="datetime", how="inner")
test_df = X_values.iloc[train_size:].merge(y_values, on="datetime", how="inner")

In [44]:
X_train, y_train = (
    train_df.drop(columns=["general_dam_occupancy_rate"]),
    train_df["general_dam_occupancy_rate"],
)

X_test, y_test = (
    test_df.drop(columns=["general_dam_occupancy_rate"]),
    test_df["general_dam_occupancy_rate"],
)


In [45]:
def get_expending_window_splitter(df: pd.DataFrame, n_fold: int = 5):
    initial_window = len(df) // 2
    step_length = (len(df) - initial_window) // n_fold
    fh = np.arange(1, step_length + 1)

    return ExpandingWindowSplitter(
        initial_window=initial_window, step_length=step_length, fh=fh
    )


## Feature Selection

In [46]:
if len(best_features) == 0:
    lgb = LGBMRegressor(importance_type="gain", random_state=SEED)
    splitter = get_expending_window_splitter(X_train)
    te = TargetEncoder(cv=splitter)
    X_train = te.fit_transform(X_train, pd.DataFrame(y_train))
    model = lgb.fit(X_train, y_train)
    feature_importance = {
        name: importance
        for name, importance in zip(model.feature_names_in_, model.feature_importances_)
    }
    number_of_features_to_select = round(len(X_train.columns) ** (1 / 2))
    best_columns = (
        pd.DataFrame(feature_importance, index=[0])
        .T.sort_values(0, ascending=False)[0:number_of_features_to_select]
        .index.to_list()
    )
    with open("models/best_features_by_lightgbm_with_te.json", "w") as f:
        json.dump(best_columns, f)


In [47]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.0001, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 10.0, log=True),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "max_bin": trial.suggest_int("max_bin", 100, 500),
        "od_type": trial.suggest_categorical("od_type", ["IncToDec", "Iter"]),
        "random_strength": trial.suggest_float("random_strength", 0.0, 100.0),
        "leaf_estimation_method": trial.suggest_categorical(
            "leaf_estimation_method", ["Newton", "Gradient"]
        ),
        "grow_policy": trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        ),
        "feature_border_type": trial.suggest_categorical(
            "feature_border_type", ["GreedyLogSum", "MinEntropy", "Uniform"]
        ),
        "one_hot_max_size": trial.suggest_int("one_hot_max_size", 2, 10),
        "max_ctr_complexity": trial.suggest_int("max_ctr_complexity", 1, 8),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "leaf_estimation_iterations": trial.suggest_int(
            "leaf_estimation_iterations", 1, 10
        ),
        "leaf_estimation_backtracking": trial.suggest_categorical(
            "leaf_estimation_backtracking", ["No", "AnyImprovement"]
        ),
        "random_score_type": trial.suggest_categorical(
            "random_score_type", ["NormalWithModelSizeDecrease", "Gumbel"]
        ),
    }

    splitter = get_expending_window_splitter(X_train)

    results = []

    for train_ind, val_ind in splitter.split(y_train):
        X_train_fold, X_val_fold = X_train.iloc[train_ind], X_train.iloc[val_ind]
        y_train_fold, y_val_fold = y_train.iloc[train_ind], y_train.iloc[val_ind]

        te_cv = get_expending_window_splitter(X_train_fold)
        te = TargetEncoder(cv=te_cv)
        X_train_fold = te.fit_transform(X_train_fold, pd.DataFrame(y_train_fold))
        X_val_fold = te.transform(X_val_fold)

        model = CatBoostRegressor(**params, random_state=SEED, early_stopping_rounds=100, logging_level="Silent")
        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        results.append(mean_absolute_error(y_val_fold, y_pred))

    dummy_forecaster_mae_error = 14.062895669528846

    return np.mean(results) if np.mean(results) < dummy_forecaster_mae_error else np.inf


In [48]:
# study = optuna.create_study(direction="minimize", storage="sqlite:///study.db")
# study.optimize(objective, n_trials=50, show_progress_bar=True, n_jobs=-1)
# best_params = study.best_params

In [49]:
# study.best_params

In [50]:
# model = CatBoostRegressor(**study.best_params, random_state=SEED, early_stopping_rounds=100)
# model.fit(X_train, y_train)

In [51]:
#joblib.dump(model, "models/catboost-2-with-te-and-cv.pkl.gz", compress="gzip")

In [52]:
model = joblib.load("models/catboost-2-with-te-and-cv.pkl.gz")

In [53]:
train_pred = model.predict(X_train)
mean_absolute_error(y_train, train_pred)

0.08612799811786172

In [54]:
test_pred = model.predict(X_test)

In [55]:
mean_absolute_error(y_test, test_pred)

0.23486103173474923

In [56]:
train_pred_series = pd.Series(train_pred, name="predictions", index=y_train.index)
test_pred_series = pd.Series(test_pred, name="predictions", index=y_test.index)

train_pred_df = pd.concat(
    [train_pred_series, y_train], axis=1
)

test_pred_df = pd.concat(
    [test_pred_series, y_test], axis=1
)


In [57]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Train", "Test"))

for i, (df, name) in enumerate(zip([train_pred_df, test_pred_df], ["Train", "Test"])):
    fig.add_trace(go.Scatter(x=df.index, y=df[TARGET], mode="lines", name=f"{name} True"), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=df.index, y=df["predictions"], mode="lines", name=f"{name} Predictions"), row=1, col=i+1)

fig.update_layout(height=400, width=1200, title_text="Predictions vs. True Values")
fig.show()
